In [22]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
import re

## Affin

In [23]:
def load_afinn(path):
    afinn = {}
    with open(path, "r") as f:
        for line in f:
            word, score = line.strip().split("\t")
            afinn[word] = int(score)
    return afinn

afinn = load_afinn("AFINN-111.txt")

In [24]:
# NEGATIONS = {"not", "no", "never", "n't"}

# def afinn_predict(text, afinn_dict):
#     words = text.lower().split()
#     score = 0
#     negate = False

#     for word in words:
#         if word in NEGATIONS:
#             negate = True
#             continue

#         word_score = afinn_dict.get(word, 0)

#         if negate:
#             word_score *= -1
#             negate = False

#         score += word_score

#     # Convert score → class
#     if score > 0:
#         return 1
#     elif score < 0:
#         return -1
#     else:
#         return 0

# -----------------------------
# Config
# -----------------------------
NEGATIONS = {"not", "no", "never", "n't"}
INTENSIFIERS = {"very": 1.5, "extremely": 2, "too": 1.5, "so": 1.3}


# -----------------------------
# Better tokenizer
# -----------------------------
def tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)


# -----------------------------
# Improved AFINN predictor
# -----------------------------
def afinn_predict(text, afinn_dict, pos_th=1, neg_th=-1):
    if pd.isna(text):
        return None

    words = tokenize(str(text))

    score = 0
    negate_window = 0
    intensity = 1

    for word in words:

        # Negation handling (window of 3 words)
        if word in NEGATIONS:
            negate_window = 3
            continue

        # Intensifiers
        if word in INTENSIFIERS:
            intensity = INTENSIFIERS[word]
            continue

        word_score = afinn_dict.get(word, 0)

        # Apply negation
        if negate_window > 0:
            word_score *= -1
            negate_window -= 1

        # Apply intensity
        word_score *= intensity
        intensity = 1  # reset

        score += word_score

    # Thresholded classification (more stable than 0 cutoff)
    if score >= pos_th:
        return 1
    elif score <= neg_th:
        return -1
    else:
        return 0

In [25]:
schemas = [
    "fully_cleaned",
    "without_lemmatization",
    "without_stopwords_lowercasing_punct"
]

for schema in schemas:
    df = pd.read_csv(f"schemas/{schema}.csv")

    df[f"afinn_pred_{schema}"] = df["cleaned_review"].apply(
        lambda x: afinn_predict(x, afinn)
    )

    df[f"afinn_pred_{schema}"] = df[f"afinn_pred_{schema}"].map({
        -1: "negative",
        0: "neutral",
        1: "positive"
    })

    df[["sentiment", f"afinn_pred_{schema}"]] \
        .to_csv(f"output/afinn_predictions_{schema}.csv", index=False)

In [26]:
df = pd.read_csv('output/afinn_predictions_fully_cleaned.csv')
print(classification_report(df['sentiment'], df['afinn_pred_fully_cleaned']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_fully_cleaned']))
print("AFINN Fully Cleaned F1 Score:", f1_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Precision:", precision_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Recall:", recall_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Accuracy Score:", accuracy_score(df['sentiment'], df['afinn_pred_fully_cleaned']))


              precision    recall  f1-score   support

    negative       0.57      0.45      0.51      1666
     neutral       0.33      0.04      0.07      1666
    positive       0.42      0.86      0.56      1664

    accuracy                           0.45      4996
   macro avg       0.44      0.45      0.38      4996
weighted avg       0.44      0.45      0.38      4996

[[ 758   92  816]
 [ 394   70 1202]
 [ 182   49 1433]]
AFINN Fully Cleaned F1 Score: 0.3800055968869356
AFINN Fully Cleaned Precision: 0.43841307395785384
AFINN Fully Cleaned Recall: 0.45256204963971175
AFINN Fully Cleaned Accuracy Score: 0.45256204963971175


In [27]:
df = pd.read_csv('output/afinn_predictions_without_lemmatization.csv')
print(classification_report(df['sentiment'], df['afinn_pred_without_lemmatization']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_without_lemmatization']))
print("AFINN Without Lemmatization F1 Score:", f1_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))
print("AFINN Without Lemmatization Precision:", precision_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))
print("AFINN Without Lemmatization Recall:", recall_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))

              precision    recall  f1-score   support

    negative       0.57      0.45      0.50      1666
     neutral       0.34      0.05      0.08      1666
    positive       0.42      0.86      0.56      1664

    accuracy                           0.45      4996
   macro avg       0.44      0.45      0.38      4996
weighted avg       0.44      0.45      0.38      4996

[[ 753  103  810]
 [ 390   77 1199]
 [ 186   47 1431]]
AFINN Without Lemmatization F1 Score: 0.38157066869227546
AFINN Without Lemmatization Precision: 0.44060546500183095
AFINN Without Lemmatization Recall: 0.45256204963971175


In [28]:
df = pd.read_csv('output/afinn_predictions_without_stopwords_lowercasing_punct.csv')
print(classification_report(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))
print("AFINN Without Stopwords, Lowercasing, Punctuation F1 Score:", f1_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Precision:", precision_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Recall:", recall_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Accuracy Score:", accuracy_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))

              precision    recall  f1-score   support

    negative       0.59      0.43      0.50      1666
     neutral       0.32      0.04      0.08      1666
    positive       0.41      0.88      0.56      1666

    accuracy                           0.45      4998
   macro avg       0.44      0.45      0.38      4998
weighted avg       0.44      0.45      0.38      4998

[[ 723  109  834]
 [ 335   74 1257]
 [ 159   46 1461]]
AFINN Without Stopwords, Lowercasing, Punctuation F1 Score: 0.3798819354656932
AFINN Without Stopwords, Lowercasing, Punctuation Precision: 0.4428484950083761
AFINN Without Stopwords, Lowercasing, Punctuation Recall: 0.45178071228491395
AFINN Without Stopwords, Lowercasing, Punctuation Accuracy Score: 0.45178071228491395


## Vader

In [29]:
import nltk

In [32]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# def vader_predict(text):
#     score = sia.polarity_scores(str(text))["compound"]

#     if score > 0.05:
#         return 1
#     elif score < -0.05:
#         return -1
#     else:
#         return 0


def vader_predict(text, pos_th=0.1, neg_th=-0.1, return_score=False):

    if not text:
        return None if not return_score else (None, 0)

    score = sia.polarity_scores(text)["compound"]

    if score >= pos_th:
        label = 1
    elif score <= neg_th:
        label = -1
    else:
        label = 0

    if return_score:
        return label, score

    return label

In [33]:
# df["vader_pred"] = df["cleaned_review"].apply(vader_predict)
schemas = ["fully_cleaned", "without_lemmatization", "without_stopwords_lowercasing_punct"]
for schema in schemas:
    df = pd.read_csv("schemas/"+schema+".csv")
    
    df[f"vader_pred_{schema}"] = df["cleaned_review"].apply(vader_predict)
    df[f"vader_pred_{schema}"] = df[f"vader_pred_{schema}"].map({-1: "negative", 0: "neutral", 1: "positive"})
    df[["sentiment", f"vader_pred_{schema}"]].to_csv(f"output/vader_predictions_{schema}.csv", index=False)

In [34]:
df = pd.read_csv('output/vader_predictions_fully_cleaned.csv')
print(classification_report(df["sentiment"], df["vader_pred_fully_cleaned"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_fully_cleaned"]))
print("VADER Fully Cleaned F1 Score:", f1_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Precision:", precision_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Recall:", recall_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_fully_cleaned']))

              precision    recall  f1-score   support

    negative       0.57      0.42      0.48      1666
     neutral       0.36      0.03      0.06      1666
    positive       0.40      0.87      0.55      1664

    accuracy                           0.44      4996
   macro avg       0.44      0.44      0.36      4996
weighted avg       0.44      0.44      0.36      4996

[[ 701   71  894]
 [ 341   56 1269]
 [ 186   30 1448]]
VADER Fully Cleaned F1 Score: 0.3648904975670331
VADER Fully Cleaned Precision: 0.44286067131425133
VADER Fully Cleaned Recall: 0.44135308246597277
VADER Fully Cleaned Accuracy Score: 0.44135308246597277


In [35]:
df = pd.read_csv('output/vader_predictions_without_lemmatization.csv')
print(classification_report(df["sentiment"], df["vader_pred_without_lemmatization"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_without_lemmatization"]))
print("VADER Without Lemmatization F1 Score:", f1_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Precision:", precision_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Recall:", recall_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_without_lemmatization']))

              precision    recall  f1-score   support

    negative       0.57      0.42      0.48      1666
     neutral       0.29      0.03      0.05      1666
    positive       0.40      0.87      0.55      1664

    accuracy                           0.44      4996
   macro avg       0.42      0.44      0.36      4996
weighted avg       0.42      0.44      0.36      4996

[[ 700   87  879]
 [ 346   48 1272]
 [ 187   30 1447]]
VADER Without Lemmatization F1 Score: 0.3617034949770815
VADER Without Lemmatization Precision: 0.4202732342518914
VADER Without Lemmatization Recall: 0.43935148118494793
VADER Without Lemmatization Accuracy Score: 0.43935148118494793


In [36]:
df = pd.read_csv('output/vader_predictions_without_stopwords_lowercasing_punct.csv')
print(classification_report(df["sentiment"], df["vader_pred_without_stopwords_lowercasing_punct"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_without_stopwords_lowercasing_punct"]))
print("VADER Without Stopwords, Lowercasing, Punctuation F1 Score:", f1_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Precision:", precision_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Recall:", recall_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct']))

              precision    recall  f1-score   support

    negative       0.58      0.45      0.51      1666
     neutral       0.33      0.04      0.07      1666
    positive       0.41      0.86      0.55      1666

    accuracy                           0.45      4998
   macro avg       0.44      0.45      0.38      4998
weighted avg       0.44      0.45      0.38      4998

[[ 752   81  833]
 [ 343   60 1263]
 [ 195   39 1432]]
VADER Without Stopwords, Lowercasing, Punctuation F1 Score: 0.3750688515965654
VADER Without Stopwords, Lowercasing, Punctuation Precision: 0.4407249204591397
VADER Without Stopwords, Lowercasing, Punctuation Recall: 0.4489795918367347
VADER Without Stopwords, Lowercasing, Punctuation Accuracy Score: 0.4489795918367347
